In [ ]:
import os, json, shutil
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score, accuracy_score, classification_report

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
BASE = r'C:\Users\Abdullah\Desktop\Code\Research Datasets\Nastaliq\Sentiment Analysis'

dfa_train = pd.read_csv(os.path.join(BASE, 'Domain_A_Twitter_Social_Media',       'train.csv'), encoding='utf-8-sig')
dfa_test  = pd.read_csv(os.path.join(BASE, 'Domain_A_Twitter_Social_Media',       'test.csv'),  encoding='utf-8-sig')
dfb_train = pd.read_csv(os.path.join(BASE, 'Domain_B_Movie_Reviews',              'train.csv'), encoding='utf-8-sig')
dfb_test  = pd.read_csv(os.path.join(BASE, 'Domain_B_Movie_Reviews',              'test.csv'),  encoding='utf-8-sig')
dfc_train = pd.read_csv(os.path.join(BASE, 'Domain_C_Civic_Social_Topics',        'train.csv'), encoding='utf-8-sig')
dfc_test  = pd.read_csv(os.path.join(BASE, 'Domain_C_Civic_Social_Topics',        'test.csv'),  encoding='utf-8-sig')
dfd_train = pd.read_csv(os.path.join(BASE, 'Domain_D_Governance_Political',       'train.csv'), encoding='utf-8-sig')
dfd_test  = pd.read_csv(os.path.join(BASE, 'Domain_D_Governance_Political',       'test.csv'),  encoding='utf-8-sig')
dfe_train = pd.read_csv(os.path.join(BASE, 'Domain_E_Socioeconomic_Agricultural', 'train.csv'), encoding='utf-8-sig')
dfe_test  = pd.read_csv(os.path.join(BASE, 'Domain_E_Socioeconomic_Agricultural', 'test.csv'),  encoding='utf-8-sig')

print('Domain sizes (train / test):')
for name, tr, te in [
    ('A_Twitter_Social_Media',       dfa_train, dfa_test),
    ('B_Movie_Reviews',              dfb_train, dfb_test),
    ('C_Civic_Social_Topics',        dfc_train, dfc_test),
    ('D_Governance_Political',       dfd_train, dfd_test),
    ('E_Socioeconomic_Agricultural', dfe_train, dfe_test),
]:
    print(f'  {name}: train={len(tr)}  test={len(te)}  labels={tr["label"].value_counts().to_dict()}')

In [ ]:
XLM_MODEL_ID  = 'xlm-roberta-base'
BERT_MODEL_ID = 'bert-base-multilingual-cased'

NUM_LABELS    = 2
MAX_LEN       = 128
MAX_TRAIN     = 8000
EPOCHS        = 3
PATIENCE      = 2
BATCH_TRAIN   = 16
BATCH_EVAL    = 32
LR            = 2e-5

RESULTS_BASE  = r'C:\Users\Abdullah\Desktop\Code\Research Datasets\results\T1_Nastaliq_SA'
os.makedirs(RESULTS_BASE, exist_ok=True)

domains = [
    ('A_Twitter_Social_Media',       dfa_train, dfa_test),
    ('B_Movie_Reviews',              dfb_train, dfb_test),
    ('C_Civic_Social_Topics',        dfc_train, dfc_test),
    ('D_Governance_Political',       dfd_train, dfd_test),
    ('E_Socioeconomic_Agricultural', dfe_train, dfe_test),
]

In [ ]:
def cap_dataset(df, max_samples=MAX_TRAIN):
    """Stratified cap — safe even if all 1s come before all 0s in the file."""
    if len(df) <= max_samples:
        return df
    capped = df.groupby('label', group_keys=False).apply(
        lambda x: x.sample(
            min(len(x), round(max_samples * len(x) / len(df))),
            random_state=42
        )
    )
    return capped.sample(frac=1, random_state=42).reset_index(drop=True)


class UrduDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.labels = df['label'].astype(int).tolist()
        self.enc = tokenizer(
            df['text'].astype(str).tolist(),
            padding='max_length',
            truncation=True,
            max_length=MAX_LEN,
            return_tensors='pt'
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.enc['input_ids'][idx],
            'attention_mask': self.enc['attention_mask'][idx],
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'macro_f1': f1_score(labels, preds, average='macro'),
        'accuracy': accuracy_score(labels, preds)
    }


def run_one(model_id, model_label, src_name, train_df, tgt_name, test_df):
    """
    One experiment: fine-tune on train_df, evaluate on test_df.
    Skips automatically if result JSON already exists.
    Returns (macro_f1, trained_trainer) — trainer is None if skipped.
    """
    out_dir     = os.path.join(RESULTS_BASE, model_label)
    os.makedirs(out_dir, exist_ok=True)
    result_path = os.path.join(out_dir, f'{src_name}__vs__{tgt_name}.json')

    if os.path.exists(result_path):
        print(f'  [SKIP] {src_name} -> {tgt_name} already done.')
        with open(result_path, encoding='utf-8') as f:
            return json.load(f)['macro_f1'], None

    run_type = 'IN-DOMAIN' if src_name == tgt_name else 'CROSS-DOMAIN'
    capped   = cap_dataset(train_df)
    print(f'\n  [{run_type}] Train: {src_name} ({len(capped)}) -> Test: {tgt_name} ({len(test_df)})')

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model     = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=NUM_LABELS)

    val_df    = capped.sample(max(int(len(capped) * 0.1), 1), random_state=42)
    train_sub = capped.drop(val_df.index)

    ckpt_dir = os.path.join(out_dir, f'_ckpt_{src_name}_{tgt_name}')

    args = TrainingArguments(
        output_dir                  = ckpt_dir,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_TRAIN,
        per_device_eval_batch_size  = BATCH_EVAL,
        learning_rate               = LR,
        warmup_ratio                = 0.1,
        weight_decay                = 0.01,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'macro_f1',
        greater_is_better           = True,
        logging_steps               = 50,
        fp16                        = torch.cuda.is_available(),
        report_to                   = 'none',
        save_total_limit            = 1,
    )

    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = UrduDataset(train_sub, tokenizer),
        eval_dataset    = UrduDataset(val_df, tokenizer),
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=PATIENCE)]
    )
    trainer.train()

    preds_out = trainer.predict(UrduDataset(test_df, tokenizer))
    preds     = np.argmax(preds_out.predictions, axis=-1)
    labels    = preds_out.label_ids

    macro_f1 = f1_score(labels, preds, average='macro')
    accuracy = accuracy_score(labels, preds)

    result = {
        'task': 'T1_Nastaliq_SA', 'model': model_label, 'model_id': model_id,
        'source': src_name, 'target': tgt_name, 'type': run_type,
        'train_size': len(capped), 'test_size': len(test_df),
        'macro_f1': round(macro_f1, 4), 'accuracy': round(accuracy, 4),
        'classification_report': classification_report(labels, preds, output_dict=True)
    }
    with open(result_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    if os.path.exists(ckpt_dir):
        shutil.rmtree(ckpt_dir)

    print(f'  macro-F1={macro_f1:.4f}  accuracy={accuracy:.4f}')
    return macro_f1, trainer


print('Helpers loaded. Ready to run experiments.')


**Model 1 — XLM-R Base**

In [ ]:
print('=== XLM-R | Source: A_Twitter_Social_Media ===')
xlmr_results = getattr(xlmr_results if 'xlmr_results' in dir() else type('', (), {})(), '__dict__', {}) or {}
xlmr_results = globals().get('xlmr_results', {})

src_name, train_df, _ = domains[0]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(XLM_MODEL_ID, 'XLM-R_Base', src_name, train_df, tgt_name, test_df)
    xlmr_results[(src_name, tgt_name)] = f1

print('\nSource A done.')

In [ ]:
print('=== XLM-R | Source: B_Movie_Reviews ===')

src_name, train_df, _ = domains[1]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(XLM_MODEL_ID, 'XLM-R_Base', src_name, train_df, tgt_name, test_df)
    xlmr_results[(src_name, tgt_name)] = f1

print('\nSource B done.')

In [ ]:
print('=== XLM-R | Source: C_Civic_Social_Topics ===')

src_name, train_df, _ = domains[2]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(XLM_MODEL_ID, 'XLM-R_Base', src_name, train_df, tgt_name, test_df)
    xlmr_results[(src_name, tgt_name)] = f1

print('\nSource C done.')

In [ ]:
print('=== XLM-R | Source: D_Governance_Political ===')

src_name, train_df, _ = domains[3]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(XLM_MODEL_ID, 'XLM-R_Base', src_name, train_df, tgt_name, test_df)
    xlmr_results[(src_name, tgt_name)] = f1

print('\nSource D done.')

In [ ]:
print('=== XLM-R | Source: E_Socioeconomic_Agricultural ===')

src_name, train_df, _ = domains[4]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(XLM_MODEL_ID, 'XLM-R_Base', src_name, train_df, tgt_name, test_df)
    xlmr_results[(src_name, tgt_name)] = f1

print('\nXLM-R — all 25 runs complete.')
print('Results so far:')
for (s, t), f1 in sorted(xlmr_results.items()):
    tag = 'IN ' if s == t else 'X  '
    print(f'  [{tag}] {s} -> {t}: {f1:.4f}')

**Model 2 — mBERT**

In [ ]:
print('=== mBERT | Source: A_Twitter_Social_Media ===')
mbert_results = globals().get('mbert_results', {})

src_name, train_df, _ = domains[0]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(BERT_MODEL_ID, 'mBERT', src_name, train_df, tgt_name, test_df)
    mbert_results[(src_name, tgt_name)] = f1

print('\nSource A done.')

In [ ]:
print('=== mBERT | Source: B_Movie_Reviews ===')

src_name, train_df, _ = domains[1]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(BERT_MODEL_ID, 'mBERT', src_name, train_df, tgt_name, test_df)
    mbert_results[(src_name, tgt_name)] = f1

print('\nSource B done.')

In [ ]:
print('=== mBERT | Source: C_Civic_Social_Topics ===')

src_name, train_df, _ = domains[2]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(BERT_MODEL_ID, 'mBERT', src_name, train_df, tgt_name, test_df)
    mbert_results[(src_name, tgt_name)] = f1

print('\nSource C done.')

In [ ]:
print('=== mBERT | Source: D_Governance_Political ===')

src_name, train_df, _ = domains[3]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(BERT_MODEL_ID, 'mBERT', src_name, train_df, tgt_name, test_df)
    mbert_results[(src_name, tgt_name)] = f1

print('\nSource D done.')

In [ ]:
print('=== mBERT | Source: E_Socioeconomic_Agricultural ===')

src_name, train_df, _ = domains[4]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(BERT_MODEL_ID, 'mBERT', src_name, train_df, tgt_name, test_df)
    mbert_results[(src_name, tgt_name)] = f1

print('\nmBERT — all 25 runs complete.')
print('Results so far:')
for (s, t), f1 in sorted(mbert_results.items()):
    tag = 'IN ' if s == t else 'X  '
    print(f'  [{tag}] {s} -> {t}: {f1:.4f}')


**Table 4 — Compute and Save Results**

In [ ]:
def compute_table4(results, model_label):
    names = [d[0] for d in domains]
    ss = [results[(d, d)] for d in names]
    st = [results[(s, t)] for s in names for t in names if s != t]
    sd_pairs = [results[(s,s)] - results[(s,t)] for s in names for t in names if s!=t]
    td_pairs = [results[(t,t)] - results[(s,t)] for s in names for t in names if s!=t]

    row = {
        'model':  model_label,
        'task':   'T1_Nastaliq_SA',
        'avg_SS': round(np.mean(ss), 4),
        'avg_ST': round(np.mean(st), 4),
        'avg_SD': round(np.mean(sd_pairs), 4),
        'WSD':    round(max(sd_pairs), 4),
        'avg_TD': round(np.mean(td_pairs), 4),
        'WTD':    round(max(td_pairs), 4),
    }
    print(f"\n{model_label}")
    print(f"  Avg In-Domain  (SS) : {row['avg_SS']}")
    print(f"  Avg Cross-Domain(ST): {row['avg_ST']}")
    print(f"  Avg Source Drop (SD): {row['avg_SD']}")
    print(f"  Worst Source Drop   : {row['WSD']}")
    print(f"  Avg Target Drop (TD): {row['avg_TD']}")
    print(f"  Worst Target Drop   : {row['WTD']}")
    return row

row_xlmr  = compute_table4(xlmr_results,  'XLM-R_Base')
row_mbert = compute_table4(mbert_results, 'mBERT')

summary_df = pd.DataFrame([row_xlmr, row_mbert])
print('\n── Summary ──')
print(summary_df.to_string(index=False))

In [ ]:
out_path = r'C:\Users\Abdullah\Desktop\Code\Research Datasets\results\Table4_T1_Nastaliq_SA.csv'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
summary_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

**HuggingFace Upload — Open Source Contribution**

In [ ]:
from huggingface_hub import HfApi, login

HF_TOKEN    = ''          # paste your HuggingFace write token here
HF_USERNAME = ''          # your HuggingFace username

login(token=HF_TOKEN)
print('Logged in to HuggingFace.')

In [ ]:
names = [d[0] for d in domains]
ss_scores = {name: xlmr_results[(name, name)] for name in names}
best_domain_name = max(ss_scores, key=ss_scores.get)
best_domain_df   = next(tr for n, tr, _ in domains if n == best_domain_name)

print(f'Best in-domain: {best_domain_name}  (F1={ss_scores[best_domain_name]:.4f})')
print(f'Re-training on full {len(best_domain_df)} samples for HuggingFace upload...')

REPO_NAME = f'{HF_USERNAME}/xlm-roberta-base-urdu-sentiment'
HF_CKPT   = os.path.join(RESULTS_BASE, 'hf_upload_model')

tokenizer_hf = AutoTokenizer.from_pretrained(XLM_MODEL_ID)
model_hf     = AutoModelForSequenceClassification.from_pretrained(
    XLM_MODEL_ID,
    num_labels=NUM_LABELS,
    id2label={0: 'NEGATIVE', 1: 'POSITIVE'},
    label2id={'NEGATIVE': 0, 'POSITIVE': 1}
)

val_hf    = best_domain_df.sample(max(int(len(best_domain_df) * 0.1), 1), random_state=42)
train_hf  = best_domain_df.drop(val_hf.index)

hf_args = TrainingArguments(
    output_dir                  = HF_CKPT,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_TRAIN,
    per_device_eval_batch_size  = BATCH_EVAL,
    learning_rate               = LR,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'macro_f1',
    greater_is_better           = True,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none',
    save_total_limit            = 1,
    push_to_hub                 = True,
    hub_model_id                = REPO_NAME,
    hub_token                   = HF_TOKEN,
)

hf_trainer = Trainer(
    model           = model_hf,
    args            = hf_args,
    train_dataset   = UrduDataset(train_hf, tokenizer_hf),
    eval_dataset    = UrduDataset(val_hf,   tokenizer_hf),
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=PATIENCE)]
)

hf_trainer.train()

hf_trainer.push_to_hub(
    commit_message='Upload XLM-R Base fine-tuned on Urdu Sentiment Analysis'
)
tokenizer_hf.push_to_hub(REPO_NAME, token=HF_TOKEN)

print(f'\nModel pushed to: https://huggingface.co/{REPO_NAME}')
print('Anyone can now use it with:')
print(f'  from transformers import pipeline')
print(f'  pipe = pipeline("text-classification", model="{REPO_NAME}")')